# Tutorial: Clean Messy Data Without Losing Track of What Broke

When you clean messy CSV files in Pandas using `pd.to_datetime(df['date'], errors='coerce')` or `pd.to_numeric(df['price'], errors='coerce')`, bad values silently turn into `NaT` or `NaN`. That works fine for quick homework, but in real projects it hides *why* a row failed—was the cell empty, in a Thai Buddhist year (`2567-01-15`), or complete garbage (`'NotDate'`)?

In this tutorial, we build a clean, undergraduate-friendly version of **`Monad`** and **`SquishyEngine`**. We strip away advanced multiprocessing, `__slots__`, and complex type-spec dictionaries so you can focus 100% on the core pattern: **chaining pure Python functions with the `|` operator while automatically recording a step-by-step audit log.**

---

## Step 0: Our 5-Row Messy Dataset

Only `pandas` is needed (`pip install pandas`). Let's start with a tiny 5-row dataset so we can trace every single cell by eye:

In [1]:
import pandas as pd

toy_df = pd.DataFrame({
    "date_col":    ["2024-01-15", "2567-01-15", "31/01/2024", None,      "NotDate"],
    "country_col": ["TH",         "USA",        "UK",         "BadCode", "n/a"    ],
    "price_col":   ["100",        "$50.00",     "1,000",      "Free",    "฿250"   ],
})
print(toy_df)

     date_col country_col price_col
0  2024-01-15          TH       100
1  2567-01-15         USA    $50.00
2  31/01/2024          UK     1,000
3        None     BadCode      Free
4     NotDate         n/a      ฿250


Each row tests a different real-world headache:

* **Row 0:** Clean standard values (`'2024-01-15'`, `'TH'`, `'100'`).
* **Row 1:** Thai Buddhist year (`2567 - 543 = 2024`), `'USA'` instead of `'US'`, and a `$` sign.
* **Row 2:** UK date format (`dd/mm/yyyy`) and a comma inside `'1,000'`.
* **Row 3:** Missing date (`None`), unknown country (`'BadCode'`), and text price (`'Free'`).
* **Row 4:** Corrupt date (`'NotDate'`), fake null string (`'n/a'`), and a Thai Baht symbol (`'฿250'`).

---

## Part 1: Writing Pure Rules (`@validator` vs. `@transformer`)

Before building the engine, we write small, single-purpose functions. Every rule falls into one of two roles:

| Role | Analogy | If It Succeeds | If It Raises an Error |
| --- | --- | --- | --- |
| **`@validator`** | **Security Checkpoint (AND)** | Let the value pass to the next step. | Reject the cell immediately (`dirty`) and **stop**. |
| **`@transformer`** | **Keyring of Parsers (OR)** | Save the cleaned value (`success`) and **stop** trying other keys. | Log the failure, then **try the next parser** in line. |

We tag functions using two tiny Python decorators:

In [2]:
def validator(func):
    func._role = "validator"
    return func

def transformer(func):
    func._role = "transformer"
    return func

Now look how clean our data rules are—no `try/except` blocks or logging clutter inside any of them:

In [3]:
@validator
def must_exist(v):
    if pd.isna(v) or str(v).strip().lower() in ["", "n/a", "null", "none", "nan"]:
        raise ValueError("Missing value")
    return v

@transformer
def to_iso_code(v):
    mapping = {"TH": "TH", "USA": "US", "UK": "UK", "JP": "JP"}
    code = str(v).strip().upper()
    if code not in mapping:
        raise ValueError(f"Unknown country: {v}")
    return mapping[code]

@transformer
def to_float(v):
    cleaned = str(v).replace(",", "").replace("$", "").replace("฿", "")
    return float(cleaned)

@transformer
def parse_iso(v):
    return pd.to_datetime(v, format="%Y-%m-%d")

@transformer
def parse_uk(v):
    return pd.to_datetime(v, format="%d/%m/%Y")

@transformer
def parse_thai(v):
    parts = str(v).split("-")
    if len(parts) == 3 and parts[0].isdigit() and int(parts[0]) > 2400:
        gregorian_year = int(parts[0]) - 543
        return pd.to_datetime(f"{gregorian_year}-{parts[1]}-{parts[2]}", format="%Y-%m-%d")
    raise ValueError("Not a Thai Buddhist date")

---

## Part 2: The Core `Monad` Class (Box + Pipe `|`)

Think of a **`Monad`** as a smart protective box around a single cell value. Instead of calling functions manually inside nested `try/except` blocks, we pipe the box through functions using Python's `|` operator (`__or__`):

Here is the simplified, readable `Monad` implementation:

In [6]:
class Monad:
    def __init__(self, value, row, col):
        self.value = value
        self.input_row = row
        self.output_column = col
        self.status = "pending"   # 'pending' -> 'valid' -> 'success' (or 'dirty')
        self.final_value = None
        self.stopped = False
        self.step = 0
        self.logs = []

    def _log(self, func_name, role_type, status, details=None):
        self.logs.append({
            "row": self.input_row,
            "col": self.output_column,
            "step": self.step,
            "role_type": role_type,
            "role": func_name,
            "status": status,
            "value": str(self.value),
            "details": details,
        })

    def __or__(self, func):
        """Overloads the `|` operator so we can write: monad | func"""
        self.step += 1
        role = getattr(func, "_role", "validator")
        name = func.__name__

        # 1. If the chain already finished or failed, mark remaining steps 'skipped'
        if self.stopped:
            self._log(name, role, "skipped")
            return self

        # 2. Otherwise, run the function inside a try/except safety net
        try:
            result = func(self.value)
            if role == "validator":
                self.value = result
                self.status = "valid"
                self._log(name, role, "passed")
            elif role == "transformer":
                self.value = result
                self.final_value = result
                self.status = "success"
                self.stopped = True  # First working parser wins! Stop trying others.
                self._log(name, role, "passed")

        except Exception as err:
            if role == "validator":
                self.status = "dirty"
                self.final_value = None
                self.stopped = True  # Failed security check! Stop immediately.
                self._log(name, role, "failed", str(err))
            elif role == "transformer":
                # Parser didn't fit—log failure, keep self.stopped = False to try the next one!
                self._log(name, role, "failed", str(err))

        return self

    def apply(self, pipeline):
        """Runs a list of functions and checks if all transformers failed."""
        has_transformer = False
        for func in pipeline:
            if getattr(func, "_role", "validator") == "transformer":
                has_transformer = True
            self | func  # Calls self.__or__(func)

        # What if 'NotDate' passed must_exist ('valid'), then failed EVERY date parser?
        if self.status in ["pending", "valid"]:
            if has_transformer:
                self.status = "dirty"
                self.final_value = None
                self.step += 1
                self._log("chain_exhausted", "system", "failed", "All transformers failed")
            else:
                # Pipeline only had validators, and all of them passed!
                self.status = "success"
                self.final_value = self.value

        return self

In [8]:
x = Monad("31/01/2024", row=2, col="clean_date") | must_exist | parse_thai | parse_uk | parse_iso
x.logs

[{'row': 2,
  'col': 'clean_date',
  'step': 1,
  'role_type': 'validator',
  'role': 'must_exist',
  'status': 'passed',
  'value': '31/01/2024',
  'details': None},
 {'row': 2,
  'col': 'clean_date',
  'step': 2,
  'role_type': 'transformer',
  'role': 'parse_thai',
  'status': 'failed',
  'value': '31/01/2024',
  'details': 'Not a Thai Buddhist date'},
 {'row': 2,
  'col': 'clean_date',
  'step': 3,
  'role_type': 'transformer',
  'role': 'parse_uk',
  'status': 'passed',
  'value': '2024-01-31 00:00:00',
  'details': None},
 {'row': 2,
  'col': 'clean_date',
  'step': 4,
  'role_type': 'transformer',
  'role': 'parse_iso',
  'status': 'skipped',
  'value': '2024-01-31 00:00:00',
  'details': None}]

### Test Drive: Watch 3 Cells Travel Through `Monad`

Before touching the whole table, let's test `Monad` on three single cells from `date_col`:

In [9]:
date_rules = [must_exist, parse_thai, parse_uk, parse_iso]

for row_id in [2, 3, 4]:
    raw_cell = toy_df.loc[row_id, "date_col"]
    m = Monad(raw_cell, row=row_id, col="clean_date").apply(date_rules)

    print(f"\nRow {row_id} ({raw_cell!r}) => status={m.status!r}, final_value={m.final_value}")
    print(pd.DataFrame(m.logs)[["step", "role_type", "role", "status", "details"]])


Row 2 ('31/01/2024') => status='success', final_value=2024-01-31 00:00:00
   step    role_type        role   status                   details
0     1    validator  must_exist   passed                      None
1     2  transformer  parse_thai   failed  Not a Thai Buddhist date
2     3  transformer    parse_uk   passed                      None
3     4  transformer   parse_iso  skipped                      None

Row 3 (None) => status='dirty', final_value=None
   step    role_type        role   status        details
0     1    validator  must_exist   failed  Missing value
1     2  transformer  parse_thai  skipped           None
2     3  transformer    parse_uk  skipped           None
3     4  transformer   parse_iso  skipped           None

Row 4 ('NotDate') => status='dirty', final_value=None
   step    role_type             role  status  \
0     1    validator       must_exist  passed   
1     2  transformer       parse_thai  failed   
2     3  transformer         parse_uk  failed   

Output:

```text
Row 2 ('31/01/2024') => status='success', final_value=2024-01-31 00:00:00
   step    role_type        role   status                   details
0     1    validator  must_exist   passed                      None
1     2  transformer  parse_thai   failed  Not a Thai Buddhist date
2     3  transformer    parse_uk   passed                      None
3     4  transformer   parse_iso  skipped                      None

Row 3 (None) => status='dirty', final_value=None
   step    role_type        role   status        details
0     1    validator  must_exist   failed  Missing value
1     2  transformer  parse_thai  skipped           None
2     3  transformer    parse_uk  skipped           None
3     4  transformer   parse_iso  skipped           None

Row 4 ('NotDate') => status='dirty', final_value=None
   step    role_type             role  status                     details
0     1    validator       must_exist  passed                        None
1     2  transformer       parse_thai  failed    Not a Thai Buddhist date
2     3  transformer         parse_uk  failed  time data "NotDate" doesn't...
3     4  transformer        parse_iso  failed  time data "NotDate" doesn't...
4     5       system  chain_exhausted  failed     All transformers failed

```

Notice how easy that is to follow:

1. `'31/01/2024'` passes `must_exist`, fails `parse_thai`, succeeds at `parse_uk`, and **skips** `parse_iso`.
2. `None` fails `must_exist` right away and **skips** all three date parsers.
3. `'NotDate'` passes `must_exist`, fails all three parsers, and gets caught by `chain_exhausted`.

---

## Part 3: `SquishyEngine` — Running `Monad` Across a DataFrame

Now that `Monad` handles a single cell, **`SquishyEngine`** simply loops over each column configuration and each row in our DataFrame, collecting two things:

1. **`self.final_df`**: The cleaned DataFrame ready for analysis.
2. **`self.logs`**: A single DataFrame containing the full step-by-step history of every cell.

In [10]:
class SquishyEngine:
    def __init__(self, config_list, source_df):
        self.config = config_list
        self.df = source_df
        self.final_df = None
        self.logs = pd.DataFrame()

    def _coerce_column(self, series, col_type):
        """Enforce clean Pandas column data types at the end."""
        if col_type == "datetime":
            return pd.to_datetime(series, errors="coerce")
        if col_type == "float":
            return pd.to_numeric(series, errors="coerce")
        return series.astype("string")

    def run(self):
        final_df = self.df.copy()
        all_logs = []

        for col_def in self.config:
            source_col = col_def["source"]
            target_col = col_def["target"]
            col_type   = col_def.get("type", "string")
            pipeline   = col_def["pipeline"]

            cleaned_values = []

            # Loop over every row clearly—no lambdas or hidden magic!
            for row_idx, raw_val in self.df[source_col].items():
                m = Monad(raw_val, row=row_idx, col=target_col).apply(pipeline)

                if m.status == "success":
                    cleaned_values.append(m.final_value)
                else:
                    cleaned_values.append(None)

                all_logs.extend(m.logs)

            final_df[target_col] = self._coerce_column(pd.Series(cleaned_values, index=self.df.index), col_type)

        self.final_df = final_df
        self.logs = pd.DataFrame(all_logs)
        return self.final_df

---

## Part 4: Putting It All Together

Declare your pipeline configuration as a plain Python list of dictionaries and run `SquishyEngine`:

In [11]:
config = [
    {
        "source": "date_col",
        "target": "clean_date",
        "type": "datetime",
        "pipeline": [must_exist, parse_thai, parse_uk, parse_iso],
    },
    {
        "source": "country_col",
        "target": "country_code",
        "type": "string",
        "pipeline": [must_exist, to_iso_code],
    },
    {
        "source": "price_col",
        "target": "price",
        "type": "float",
        "pipeline": [must_exist, to_float],
    },
]

engine = SquishyEngine(config, toy_df)
clean_df = engine.run()

print("=== 1. CLEANED DATA ===")
print(clean_df[["clean_date", "country_code", "price"]])

print("\n=== 2. QA SUMMARY (STEP COUNTS) ===")
summary = engine.logs.groupby(["col", "step", "role", "status"]).size().reset_index(name="count")
print(summary)

print("\n=== 3. FAILED CELLS REPORT ===")
failures = engine.logs[engine.logs["status"] == "failed"][["row", "col", "role", "value", "details"]]
print(failures)

=== 1. CLEANED DATA ===
  clean_date country_code   price
0 2024-01-15           TH   100.0
1 2024-01-15           US    50.0
2 2024-01-31           UK  1000.0
3        NaT         <NA>     NaN
4        NaT         <NA>   250.0

=== 2. QA SUMMARY (STEP COUNTS) ===
             col  step             role   status  count
0     clean_date     1       must_exist   failed      1
1     clean_date     1       must_exist   passed      4
2     clean_date     2       parse_thai   failed      3
3     clean_date     2       parse_thai   passed      1
4     clean_date     2       parse_thai  skipped      1
5     clean_date     3         parse_uk   failed      2
6     clean_date     3         parse_uk   passed      1
7     clean_date     3         parse_uk  skipped      2
8     clean_date     4        parse_iso   failed      1
9     clean_date     4        parse_iso   passed      1
10    clean_date     4        parse_iso  skipped      3
11    clean_date     5  chain_exhausted   failed      1
12  cou

### Output

```text
=== 1. CLEANED DATA ===
  clean_date country_code   price
0 2024-01-15           TH   100.0
1 2024-01-15           US    50.0
2 2024-01-31           UK  1000.0
3        NaT         <NA>     NaN
4        NaT         <NA>   250.0

=== 3. FAILED CELLS REPORT ===
    row           col             role       value                   details
1     0    clean_date       parse_thai  2024-01-15  Not a Thai Buddhist date
2     0    clean_date         parse_uk  2024-01-15  time data "2024-01-15"...
9     2    clean_date       parse_thai  31/01/2024  Not a Thai Buddhist date
12    3    clean_date       must_exist        None             Missing value
17    4    clean_date       parse_thai     NotDate  Not a Thai Buddhist date
18    4    clean_date         parse_uk     NotDate  time data "NotDate"...
19    4    clean_date        parse_iso     NotDate  time data "NotDate"...
20    4    clean_date  chain_exhausted     NotDate   All transformers failed
27    3  country_code      to_iso_code     BadCode  Unknown country: BadCode
29    4  country_code       must_exist         n/a             Missing value
38    3         price         to_float        Free  could not convert string...

```

---

## Quick Student Discussion / Lab Exercises

Look at **Row 0** in the Failed Cells Report above: `'2024-01-15'` is a clean ISO date, yet it triggered **two failed exceptions** (`parse_thai` and `parse_uk`) before `parse_iso` succeeded!

1. **Exercise 1 (Pipeline Ordering):** If 90% of your company's data arrives in ISO format (`YYYY-MM-DD`), how should you reorder `"pipeline": [must_exist, parse_thai, parse_uk, parse_iso]` so Python does the least work?
2. **Exercise 2 (Write a Validator):** Write a new `@validator` called `must_be_positive(v)` and place it *Wait—can you place a validator AFTER `to_float`?* Check `Monad.__or__`! Once `to_float` succeeds, `self.stopped` becomes `True`. How would you write a `@validator` or modify `to_float` so negative prices like `'-50'` are rejected as `'dirty'`?